In [1]:
import sys
from pathlib import Path

# Make the src/ package importable from within notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config.paths import check_setup, ASTEROID_CATALOG_CSV, CLOSE_APPROACH_CSV, RAW_DIR

check_setup()

BASE_DIR: C:\Users\SATYAM\Desktop\ISM_Assignments\neo-forecasting

Checking for raw data files:
  [FOUND] C:\Users\SATYAM\Desktop\ISM_Assignments\neo-forecasting\python\data\raw\near_earth_asteroids_2025.csv
  [FOUND] C:\Users\SATYAM\Desktop\ISM_Assignments\neo-forecasting\python\data\raw\asteroid_close_approaches_2015_2035.csv


True

In [2]:
import pandas as pd

asteroids = pd.read_csv(ASTEROID_CATALOG_CSV, low_memory=False)
approaches = pd.read_csv(CLOSE_APPROACH_CSV, low_memory=False)

print(f"Asteroid catalog: {asteroids.shape}")
print(f"Close-approach events: {approaches.shape}")
asteroids.head()

Asteroid catalog: (41281, 29)
Close-approach events: (27430, 13)


,spkid,full_name,pdes,name,pha,H,diameter_km,diameter_m,diameter_is_estimated,size_category,...,per_y,moid_au,moid_km,moid_lunar_distances,n,condition_code,first_obs,last_obs,data_arc,data_arc_years
0,20000433,433 Eros (A898 PA),433,Eros,False,10.39,16.84000,16840.0,False,Large (>1 km) — City killer+,...,1.76,0.1480,22140485.0,57.60,0.5598,0.0,1893-10-29,2021-05-13,46582.0,127.53
1,20000719,719 Albert (A911 TB),719,Albert,False,15.59,2.70683,2706.8,True,Large (>1 km) — City killer+,...,4.28,0.2010,30069172.0,78.22,0.2302,0.0,1911-10-04,2026-03-20,41806.0,114.46
2,20000887,887 Alinda (A918 AA),887,Alinda,False,13.82,4.20000,4200.0,False,Large (>1 km) — City killer+,...,3.89,0.0797,11922950.0,31.02,0.2533,0.0,1918-02-09,2025-08-27,39281.0,107.55
3,20001036,1036 Ganymed (A924 UB),1036,Ganymed,False,9.17,37.67500,37675.0,False,Large (>1 km) — City killer+,...,4.35,0.3430,51312070.0,133.49,0.2266,0.0,1924-10-23,2026-03-22,37040.0,101.41
4,20001221,1221 Amor (1932 EA1),1221,Amor,False,17.37,1.00000,1000.0,False,Large (>1 km) — City killer+,...,2.66,0.1080,16156570.0,42.03,0.3705,0.0,1932-03-12,2025-02-19,33947.0,92.94


In [3]:
approaches.head()

,designation,full_name,close_approach_date,distance_au,dist_km,dist_lunar,distance_min_au,distance_max_au,velocity_km_s,v_rel_kmh,velocity_infinity_km_s,absolute_magnitude,is_future
0,2022 AP1,(2022 AP1),2015-01-01 00:27,0.045100,6746806.0,17.55,0.013778,0.076571,11.891749,42810.0,11.886780,28.39,False
1,2015 AE45,(2015 AE45),2015-01-02 15:56,0.048239,7216474.0,18.77,0.048220,0.048258,7.065686,25436.0,7.057864,25.30,False
2,613286,613286 (2005 YQ96),2015-01-02 21:46,0.026522,3967664.0,10.32,0.026522,0.026522,12.703378,45732.0,12.695467,20.63,False
3,2014 YQ34,(2014 YQ34),2015-01-03 13:29,0.079692,11921722.0,31.01,0.078275,0.081108,24.982094,89936.0,24.980756,24.16,False
4,2014 YE42,(2014 YE42),2015-01-03 15:00,0.010995,1644896.0,4.28,0.010971,0.011020,13.998882,50396.0,13.981561,23.40,False


In [4]:
from src.data.fetch_jpl import fetch_sbdb_record, sbdb_elements_to_dict

# Example: fetch the LATEST Apophis orbital solution directly from JPL
# (uncomment to run -- requires internet access)

apophis_live = fetch_sbdb_record("99942", covariance=True, phys_par=True)
flat = sbdb_elements_to_dict(apophis_live)
print("Latest orbital elements:", flat["values"])
print("Latest sigmas:", flat["sigmas"])
print("Epoch (JD):", flat["epoch_jd"])

Latest orbital elements: {'e': 0.1911492279663492, 'a': 0.9223592206975018, 'q': 0.7460509677535309, 'i': 3.340996879880978, 'om': 203.8936514240762, 'w': 126.6795706895841, 'ma': 175.3304026592739, 'tp': 2461042.919201488, 'per': 323.5553366891694, 'n': 1.112638115271892, 'ad': 1.098667473641473}
Latest sigmas: {'e': 1.8376e-09, 'a': 1.5667e-09, 'q': 2.8517e-09, 'i': 9.903e-08, 'om': 3.0657e-06, 'w': 3.3086e-06, 'ma': 5.2809e-07, 'tp': 7.0729e-07, 'per': 8.244e-07, 'n': 2.8349e-09, 'ad': 1.8662e-09}
Epoch (JD): 2461200.5


In [5]:
from src.data.fetch_cneos import fetch_sentry_table

# Example: fetch JPL's current Sentry Risk Table (objects with non-zero
# impact probability) -- useful for validating/comparing our own risk
# scoring against NASA's real assessments.
# (uncomment to run -- requires internet access)

sentry_df = fetch_sentry_table()
print(f"Objects currently on Sentry table: {len(sentry_df)}")
sentry_df.head(10)

Objects currently on Sentry table: 2155


,ts_max,des,last_obs,ps_max,ip,id,n_imp,v_inf,ps_cum,range,diameter,fullname,last_obs_jd,h
0,0,1979 XB,1979-12-15,-2.99,8.515158e-07,bJ79X00B,4,23.7606234552547,-2.69,2056-2113,0.66,(1979 XB),2444222.5,18.54
1,0,2022 KK2,2022-05-23,-5.78,1.203298e-04,bK22K02K,33,15.5694051293592,-5.58,2060-2122,0.0069,(2022 KK2),2459722.5,28.45
2,0,2000 SG344,2000-10-03,-3.11,2.743395e-03,bK00SY4G,300,1.35802744453748,-2.77,2069-2122,0.037,(2000 SG344),2451820.5,24.79
3,0,2012 VS76,2012-11-16,-6.05,1.944201e-05,bK12V76S,15,11.4626328606267,-5.74,2081-2120,0.014,(2012 VS76),2456247.5,26.95
4,0,2018 GN,2018-04-09,-8.95,3.772000e-09,bK18G00N,1,20.20,-8.95,2102-2102,0.02,(2018 GN),2458217.5,26.19
5,0,2011 TO,2013-01-12,-6.16,2.937000e-06,bK11T00O,1,8.55,-6.16,2064-2064,0.018,(2011 TO),2456304.5,26.32
6,0,2012 BA102,2012-02-21,-6.27,1.225221e-05,bK12BA2A,20,7.21203952320587,-6.03,2103-2122,0.017,(2012 BA102),2455978.5,26.51
7,0,2014 HN197,2014-04-27,-5.98,6.535100e-09,bK14HJ7N,12,21.9988508209515,-5.51,2069-2118,0.35,(2014 HN197),2456774.5,19.92
8,0,2005 UL6,2005-11-22,-6.95,9.900880e-08,bK05U06L,6,12.3834503195676,-6.70,2068-2119,0.042,(2005 UL6),2453696.5,24.51
9,0,2009 DV,2009-02-24,-7.37,3.101800e-08,bK09D00V,2,22.0982648784577,-7.24,2044-2102,0.036,(2009 DV),2454886.5,24.86


In [6]:
from src.data.preprocess import clean_and_merge, integrity_report
from src.config.paths import MERGED_CSV, MERGED_PARQUET

merged = clean_and_merge(asteroids, approaches)
print(f"Merged dataset shape: {merged.shape}")
report = integrity_report(merged)

Merged dataset shape: (27422, 49)
=== Data integrity report ===
  n_rows: 27422
  n_train_is_future_false: 23537
  n_test_is_future_true: 3885
  train_hazardous_rate: 0.03670816161787824
  test_hazardous_rate: 0.12998712998713
  leakage_rule_match_rate: 0.9988695208226971


In [7]:
# Save for downstream notebooks. Falls back to CSV if pyarrow/fastparquet
# aren't installed (parquet requires one of those packages).
try:
    merged.to_parquet(MERGED_PARQUET, index=False)
    print(f"Saved: {MERGED_PARQUET}")
except Exception as e:
    print(f"[INFO] Parquet save skipped ({e}). Install pyarrow for parquet support:")
    print("  pip install pyarrow")

merged.to_csv(MERGED_CSV, index=False)
print(f"Saved: {MERGED_CSV}")

Saved: C:\Users\SATYAM\Desktop\ISM_Assignments\neo-forecasting\python\data\processed\merged_dataset.parquet
Saved: C:\Users\SATYAM\Desktop\ISM_Assignments\neo-forecasting\python\data\processed\merged_dataset.csv
